# Refine clustering for HSCs (to split off P09 specific cells)

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import warnings 
import numpy as np
import pandas as pd
import seaborn as sns
import muon as mu
import anndata as ad
import gseapy as gp


warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, facecolor='white')

In [ ]:
mdata = mu.read_h5mu("17052025_celltypev3_mutations.h5mu")

In [ ]:
mdata

In [ ]:
mu.pp.intersect_obs(mdata)

In [ ]:
mdata

In [ ]:
hsc = mdata[mdata['rna'].obs['celltype_v2'].isin(['HSC','MPP'])].copy()


In [ ]:
hsc_ori = mdata[mdata['rna'].obs['celltype_v2'].isin(['HSC','MPP'])].copy()

## GEX only

In [ ]:
sc.pl.umap(hsc['rna'], color=['celltype_v3'], legend_loc='on data')

In [ ]:
sc.pp.highly_variable_genes(hsc['rna'],batch_key='dataset')

In [ ]:
sc.tl.pca(hsc['rna'])

In [ ]:
sc.pl.pca_variance_ratio(hsc['rna'])

In [ ]:
sc.pl.pca(
    hsc["rna"],
    color=["patient_alias", "patient_alias", "celltype_v3", "celltype_v3"],
    dimensions=[(0, 1), (2, 3), (0, 1), (2, 3)],
    ncols=2,
    size=2,
)

In [ ]:
sc.pp.neighbors(hsc['rna'])

In [ ]:
sc.tl.umap(hsc['rna'])

In [ ]:
sc.tl.leiden(hsc['rna'], flavor='igraph', n_iterations=-1)

In [ ]:
sc.pl.umap(hsc['rna'], color=['leiden','patient_alias','celltype_v3'], legend_loc='on data')

In [ ]:
sc.pl.umap(hsc['rna'], color=['patient_alias','timepoint'],)

## GEX and CITEseq WNN

In [ ]:
sc.pp.neighbors(hsc['rna'])
sc.pp.neighbors(hsc['protein'])

mu.pp.neighbors(hsc, key_added='wnn')

In [ ]:
mu.tl.umap(hsc, neighbors_key='wnn', random_state=10)

In [ ]:
mu.pl.umap(hsc, color=['rna:mod_weight', 'protein:mod_weight'], cmap='magma')

In [ ]:
sc.tl.leiden(hsc, resolution=1.0, neighbors_key='wnn', key_added='leiden_wnn')

In [ ]:
sc.pl.umap(hsc, color=['leiden_wnn','rna:celltype_v3','rna:patient_alias'], legend_loc='on data')

In [ ]:
sc.pl.umap(hsc[hsc['rna'].obs['patient_alias'] == 'P09'], color=['rna:timepoint','leiden_wnn'], legend_loc='on data')

In [ ]:
hsc

In [ ]:
hsc['rna'].obsm['X_umap_reclustered'] = hsc['rna'].obsm['X_umap'].copy()
hsc['rna'].obsm['X_umap_ori'] = hsc_ori['rna'].obsm['X_umap'].copy()
hsc

In [ ]:
mu.pl.embedding(hsc, basis='rna:X_umap_ori', color=['leiden_wnn'], groups=['0','1'], )

In [ ]:
mdata.obs['leiden_wnn'] = hsc[hsc['rna'].obs['patient_alias'] == 'P09'].obs['leiden_wnn'].copy()

In [ ]:
mu.pl.embedding(mdata, basis='rna:X_umap', color=['leiden_wnn'], )